## 1. Clone Repository

This cell clones the original GitHub repository into the Colab environment and enters the repository folder.

In [ ]:
import os
repo_dir = '/content/driver-drowsiness-detection'
if not os.path.exists(repo_dir):
    !git clone https://github.com/SyedAnasAhmed2004/driver-drowsiness-detection.git "{repo_dir}"
else:
    print('Repository already cloned:', repo_dir)
os.chdir(repo_dir)
print('Working directory:', os.getcwd())


## 2. Apply Modified Files

Overwrite the repository files with the Colab-compatible training and dataset sampling updates.

In [ ]:
%%bash
cat > /content/driver-drowsiness-detection/config.py <<'EOF'
"""
Central configuration for Driver Distraction Detection project.
Edit these values to match your environment before running.
"""

CONFIG = {
    # ── Paths ──────────────────────────────────────────────────────────────
    # Download the State Farm dataset from Kaggle and point this to the folder
    # that contains a 'train/' sub-directory with class sub-folders (c0..c9).
    "data_dir":        "data/",
    "checkpoint_dir":  "checkpoints/",
    "results_dir":     "results/",

    # ── Model ──────────────────────────────────────────────────────────────
    "num_classes":     10,          # c0 (safe) … c9 (10 distraction classes)
    "img_size":        224,         # MobileNetV2 / ResNet standard input size
    "backbone":        "mobilenetv2",  # choices: mobilenetv2 | resnet50

    # ── Training ───────────────────────────────────────────────────────────
    "epochs":                  20,
    "batch_size":              32,
    "learning_rate":           1e-4,
    "val_split":               0.2,   # fraction of train set held out for val
    "use_subset":              False,
    "subset_samples_per_class":2000,
    "random_seed":             42,

    # ── Inference ──────────────────────────────────────────────────────────
    "confidence_threshold": 0.6,    # below this → show "Uncertain" warning
}

# Human-readable labels for all 10 State Farm classes
CLASS_NAMES = [
    "c0: Safe driving",
    "c1: Texting (right hand)",
    "c2: Phone call (right hand)",
    "c3: Texting (left hand)",
    "c4: Phone call (left hand)",
    "c5: Radio / adjusting stereo",
    "c6: Drinking",
    "c7: Reaching behind",
    "c8: Hair / makeup",
    "c9: Talking to passenger",
]

# Alert severity per class (used by the real-time detector)
# 0 = safe  |  1 = mild distraction  |  2 = high distraction
SEVERITY = {
    0: 0,   # safe driving
    1: 2,   # texting  – high risk
    2: 2,   # phone call
    3: 2,   # texting
    4: 2,   # phone call
    5: 1,   # stereo
    6: 1,   # drinking
    7: 2,   # reaching behind
    8: 1,   # grooming
    9: 1,   # talking to passenger
}

SEVERITY_COLOR = {
    0: (0, 220, 80),    # green  – BGR for OpenCV
    1: (0, 165, 255),   # orange
    2: (0, 0, 220),     # red
}
EOF


In [ ]:
%%bash
cat > /content/driver-drowsiness-detection/train.py <<'EOF'
"""
Driver Distraction Detection - Training Script
Dataset: State Farm Distracted Driver Detection (Kaggle)
Model: MobileNetV2 Transfer Learning (fine-tuned)
"""

import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
from collections import defaultdict

import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader, random_split, Subset

from config import CONFIG
from model import DistractionDetector
from utils import save_checkpoint, plot_training_history, plot_confusion_matrix

# ─────────────────────────────────────────────
# 1. DATA TRANSFORMS
# ─────────────────────────────────────────────
def get_transforms():
    train_transform = transforms.Compose([
        transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ColorJitter(brightness=0.3, contrast=0.3, saturation=0.2),
        transforms.RandomAffine(degrees=0, translate=(0.1, 0.1)),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    val_transform = transforms.Compose([
        transforms.Resize((CONFIG["img_size"], CONFIG["img_size"])),
        transforms.ToTensor(),
        transforms.Normalize(mean=[0.485, 0.456, 0.406],
                             std=[0.229, 0.224, 0.225])
    ])
    return train_transform, val_transform

def _get_base_dataset(transform):
    return datasets.ImageFolder(
        root=os.path.join(CONFIG["data_dir"], "train"),
        transform=transform
    )

def _sample_balanced_subset(dataset, samples_per_class, seed=42):
    class_to_indices = defaultdict(list)
    for idx, (_, label) in enumerate(dataset.samples):
        class_to_indices[label].append(idx)

    rng = np.random.RandomState(seed)
    selected_indices = []
    for label, indices in class_to_indices.items():
        if len(indices) < samples_per_class:
            raise ValueError(
                f"Class {dataset.classes[label]} has only {len(indices)} samples, "
                f"cannot select {samples_per_class} without replacement."
            )
        selected_indices.extend(rng.choice(indices, samples_per_class, replace=False).tolist())

    rng.shuffle(selected_indices)
    return Subset(dataset, selected_indices)

# ─────────────────────────────────────────────
# 2. LOAD DATASET
# ─────────────────────────────────────────────
def load_data(use_subset: bool = False, samples_per_class: int = None, seed: int = None):
    train_transform, val_transform = get_transforms()
    seed = seed if seed is not None else CONFIG["random_seed"]
    samples_per_class = samples_per_class if samples_per_class is not None else CONFIG["subset_samples_per_class"]

    full_dataset = _get_base_dataset(train_transform)
    if use_subset:
        full_dataset = _sample_balanced_subset(full_dataset, samples_per_class, seed)

    val_size = int(len(full_dataset) * CONFIG["val_split"])
    train_size = len(full_dataset) - val_size
    train_dataset, val_dataset = random_split(
        full_dataset, [train_size, val_size], generator=torch.Generator().manual_seed(seed)
    )

    # Apply val transform to validation subset
    base_dataset = getattr(val_dataset.dataset, "dataset", val_dataset.dataset)
    base_dataset.transform = val_transform

    train_loader = DataLoader(
        train_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=True,
        num_workers=0,
        pin_memory=False
    )
    val_loader = DataLoader(
        val_dataset,
        batch_size=CONFIG["batch_size"],
        shuffle=False,
        num_workers=0,
        pin_memory=False
    )

    class_names = getattr(full_dataset, "classes", None)
    if class_names is None:
        class_names = full_dataset.dataset.classes

    print(f"[INFO] Classes: {class_names}")
    print(f"[INFO] Train samples: {train_size} | Val samples: {val_size}")
    return train_loader, val_loader, class_names

# ─────────────────────────────────────────────
# 3. TRAINING LOOP
# ─────────────────────────────────────────────
def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    running_loss, correct, total = 0.0, 0, 0

    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()
        outputs = model(images)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()

        running_loss += loss.item() * images.size(0)
        _, predicted = outputs.max(1)
        total += labels.size(0)
        correct += predicted.eq(labels).sum().item()

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc

def validate(model, loader, criterion, device):
    model.eval()
    running_loss, correct, total = 0.0, 0, 0
    all_preds, all_labels = [], []

    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            running_loss += loss.item() * images.size(0)
            _, predicted = outputs.max(1)
            total += labels.size(0)
            correct += predicted.eq(labels).sum().item()

            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    epoch_loss = running_loss / total
    epoch_acc = 100.0 * correct / total
    return epoch_loss, epoch_acc, all_preds, all_labels

# ─────────────────────────────────────────────
# 4. MAIN TRAINING ENTRY POINT
# ─────────────────────────────────────────────
def main():
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"[INFO] Using device: {device}")

    os.makedirs(CONFIG["checkpoint_dir"], exist_ok=True)
    os.makedirs(CONFIG["results_dir"], exist_ok=True)

    train_loader, val_loader, class_names = load_data(
        use_subset=CONFIG["use_subset"],
        samples_per_class=CONFIG["subset_samples_per_class"],
        seed=CONFIG["random_seed"]
    )

    model = DistractionDetector(num_classes=CONFIG["num_classes"]).to(device)
    print(f"[INFO] Model: {model.__class__.__name__} loaded")

    criterion = nn.CrossEntropyLoss(label_smoothing=0.1)
    optimizer = optim.Adam(model.parameters(), lr=CONFIG["learning_rate"],
                           weight_decay=1e-4)
    scheduler = ReduceLROnPlateau(optimizer, mode="min", patience=3,
                                  factor=0.5)

    history = {"train_loss": [], "val_loss": [],
               "train_acc": [], "val_acc": []}
    best_val_acc = 0.0

    print("\n[INFO] Starting training...\n")
    for epoch in range(1, CONFIG["epochs"] + 1):
        train_loss, train_acc = train_one_epoch(
            model, train_loader, criterion, optimizer, device)
        val_loss, val_acc, preds, labels = validate(
            model, val_loader, criterion, device)
        scheduler.step(val_loss)

        history["train_loss"].append(train_loss)
        history["val_loss"].append(val_loss)
        history["train_acc"].append(train_acc)
        history["val_acc"].append(val_acc)

        print(f"Epoch [{epoch:02d}/{CONFIG['epochs']}] "
              f"Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.2f}% | "
              f"Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.2f}%")

        if val_acc > best_val_acc:
            best_val_acc = val_acc
            save_checkpoint(model, optimizer, epoch, val_acc,
                            path=os.path.join(CONFIG["checkpoint_dir"], "best_model.pth"))
            print(f"  >> Best model saved (Val Acc: {val_acc:.2f}%)")

    print(f"\n[DONE] Best Validation Accuracy: {best_val_acc:.2f}%\n")

    # Final evaluation report
    _, _, final_preds, final_labels = validate(model, val_loader, criterion, device)
    print("\n[INFO] Classification Report:")
    print(classification_report(final_labels, final_preds,
                                target_names=class_names))

    plot_training_history(history,
                          save_path=os.path.join(CONFIG["results_dir"], "training_history.png"))
    plot_confusion_matrix(final_labels, final_preds, class_names,
                          save_path=os.path.join(CONFIG["results_dir"], "confusion_matrix.png"))

if __name__ == "__main__":
    main()
EOF

## 3. Mount Google Drive

Mount Google Drive so checkpoints and results persist after the Colab session ends.

In [ ]:
from google.colab import drive, files
import os

drive.mount('/content/drive')

KAGGLE_JSON_DRIVE = '/content/drive/MyDrive/kaggle.json'
KAGGLE_JSON_HOME = '/root/.kaggle/kaggle.json'
os.makedirs(os.path.dirname(KAGGLE_JSON_HOME), exist_ok=True)

if os.path.exists(KAGGLE_JSON_DRIVE):
    !cp '{KAGGLE_JSON_DRIVE}' '{KAGGLE_JSON_HOME}'
    !chmod 600 '{KAGGLE_JSON_HOME}'
    print('Copied kaggle.json from Google Drive.')
else:
    print('kaggle.json not found in Google Drive. Please upload it below.')
    uploaded = files.upload()
    if 'kaggle.json' in uploaded:
        with open(KAGGLE_JSON_HOME, 'wb') as f:
            f.write(uploaded['kaggle.json'])
        !chmod 600 '{KAGGLE_JSON_HOME}'
        print('Uploaded kaggle.json successfully.')
    else:
        raise FileNotFoundError('kaggle.json is required to download the dataset.')


## 4. Install Dependencies

Install the Python packages required by the repository, including Kaggle.

In [ ]:
!pip install -q --upgrade pip
!pip install -q -r requirements.txt kaggle


## 5. Download Dataset

Download the State Farm dataset from Kaggle and extract it.

In [ ]:
import os

DATA_ZIP = '/content/imgs.zip'
DATA_ROOT = '/content/imgs'

if not os.path.exists(DATA_ZIP):
    !kaggle competitions download -c state-farm-distracted-driver-detection -f imgs.zip -p /content/
else:
    print('Dataset archive already downloaded:', DATA_ZIP)

if not os.path.isdir(DATA_ROOT):
    !unzip -q /content/imgs.zip -d /content/
else:
    print('Dataset already extracted in /content/imgs')

print('Data root contents:')
!find /content/imgs -maxdepth 2 -type d | sed 's|^|  |'

## 6. Train Model

Update the configuration for Colab, load the subset, and train the model.

In [ ]:
import os
from config import CONFIG

CONFIG['data_dir'] = '/content/imgs'
CONFIG['checkpoint_dir'] = '/content/drive/MyDrive/driver_drowsiness/checkpoints'
CONFIG['results_dir'] = '/content/drive/MyDrive/driver_drowsiness/results'
CONFIG['use_subset'] = True
CONFIG['subset_samples_per_class'] = 2000
CONFIG['random_seed'] = 42
CONFIG['epochs'] = 20

os.makedirs(CONFIG['checkpoint_dir'], exist_ok=True)
os.makedirs(CONFIG['results_dir'], exist_ok=True)

print('CONFIG updated for Colab:')
for key in ['data_dir', 'checkpoint_dir', 'results_dir', 'use_subset', 'subset_samples_per_class', 'random_seed', 'epochs']:
    print(f'  {key}: {CONFIG[key]}')


In [ ]:
import torch
print('PyTorch version:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('CUDA device:', torch.cuda.get_device_name(0))

from train import load_data

train_loader, val_loader, class_names = load_data(use_subset=True, seed=CONFIG['random_seed'])
print('Loaded balanced dataset with', len(train_loader.dataset) + len(val_loader.dataset), 'total samples')
print('Train batches:', len(train_loader), 'Val batches:', len(val_loader))


In [ ]:
from train import main

# Run training end-to-end.
# The training loop will save the best model checkpoint and plot files to Google Drive.
main()


## 7. Save Results

List saved checkpoints and generated result files in Google Drive.

In [ ]:
!echo 'Saved checkpoints:'
!find /content/drive/MyDrive/driver_drowsiness/checkpoints -maxdepth 1 -type f | sed 's|^|  |' || true
!echo '
Saved results:'
!find /content/drive/MyDrive/driver_drowsiness/results -maxdepth 1 -type f | sed 's|^|  |' || true
